In [2]:
# Scientific computing.
import pandas as pd
import numpy as np
from collections import Counter
from scipy.sparse import lil_matrix, csr_matrix

# Machine learning.
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    confusion_matrix,
)
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTETomek

# File handling.
from pathlib import Path
import json

In [3]:
# Anchor to the notebook's location to not hardcode paths.
Notebook_Dir = Path().resolve()
Project_Root = Notebook_Dir.parents[1]
Data_Dir = Project_Root / "Clean_Data_Resources"

# Load the parquet from Text_Parsing.ipynb.
Survey_df = pd.read_parquet(Data_Dir / "Survey_df_Text_Parsed.parquet")

# Load scale guide that maps each column/semester/discipline to its scale type.
# From Organization.ipynb.
Likert_Guide_df = pd.read_csv(Data_Dir / "Survey_Results_Likert_Guide.csv")

# Load column metadata.
# From Organization.ipynb
with open(Data_Dir / "column_metadata.json", "r") as f:
    Column_Metadata = json.load(f)

In [4]:
# Paste in Likert scales, for sanity.
# There are two different likerts that are used, where the interpretations are slightly different. 
agreement_scale = {
    'Strongly agree': 5,
    'Moderately agree': 4,
    'Neither disagree nor agree': 3,
    'Moderately disagree': 2,
    'Strongly disagree': 1,
    'Unable to judge': None
}
# Agreement with statements is not the same as assessing helpfulness directly. 
# "Moderately helpful" is not the same as "neither disagree nor agree".
helpfulness_scale = {
    'Extremely helpful': 5,
    'Very helpful': 4,
    'Moderately helpful': 3,
    'Slightly helpful': 2,
    'Not at all helpful': 1,
    'Unable to judge': None
}
# Combine both mappings in a dictionary.
rating_encoding = {**agreement_scale, **helpfulness_scale}

## Step one: Primary Pairs.
T_ Columns are paired with matching R_ columns, based on theme. 
 For instance: The open response column with understanding collaboration is paired with the equivalent rating column that also measures collaborative activities. 

In [5]:
# Leader rating columns for averaging.
Leader_R_Cols = [
    "R_Leader_Helps_Understanding_encoded",
    "R_Leader_Subject_Competence_encoded",
    "R_Leader_Has_Plan_encoded",
    "R_Leader_Willing_To_Help_encoded",
]

# T_ column to its matched R_ column.
Primary_Pairs = [
    ("T_Collaboration_Understanding",      "R_Collaborative_Activities_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Helps_Understanding_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Subject_Competence_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Has_Plan_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Willing_To_Help_encoded"),
    ("T_Leader_Performance_Suggestions",   "R_Leader_Average_encoded"),
    ("T_Other_Suggestions",                "R_Overall_Session_Helpfulness_encoded"),
    ("T_Program_Overall_Suggestions",      "R_Overall_Session_Helpfulness_encoded"),
]

In [6]:
# Create an average leader score metric.and
# T_Leader_Performance_Suggestions corresponds with columns that rate leader performance.
Survey_df["R_Leader_Average_encoded"] = Survey_df[Leader_R_Cols].mean(axis=1)

In [7]:
# Build Agreement-scale row index per R_ column from Likert_Guide_df.
# Agreement Likerts are the vast majority of responses. Helpfulness Likerts aren't.

def get_agreement_index(r_col):
    # Strip _encoded suffix to match Likert_Guide_df column names.
    col_name = r_col.replace("_encoded", "")
    if col_name == "R_Leader_Average":
        # For averaged column, intersect Agreement rows across all four leader columns.
        indices = None
        for leader_col in Leader_R_Cols:
            agreement_rows = Likert_Guide_df[
                (Likert_Guide_df["Column"] == leader_col.replace("_encoded", "")) &
                (Likert_Guide_df["Scale"] == "Agreement")
            ][["Discipline", "Course_Code", "Semester", "Year"]]
            mask = Survey_df[["Discipline", "Course_Code", "Semester", "Year"]].merge(
                agreement_rows, how="inner"
            ).index
            indices = mask if indices is None else indices.intersection(mask)
        return indices
    # For single columns, filter directly.
    agreement_rows = Likert_Guide_df[
        (Likert_Guide_df["Column"] == col_name) &
        (Likert_Guide_df["Scale"] == "Agreement")
    ][["Discipline", "Course_Code", "Semester", "Year"]]
    mask = Survey_df.merge(
        agreement_rows,
        on=["Discipline", "Course_Code", "Semester", "Year"],
        how="inner"
    ).index
    return mask

In [8]:
# Bag of words builder.
def build_bow(token_lists):
    # Build vocabulary from all token lists.
    vocab = sorted(set(tok for tokens in token_lists for tok in tokens))
    vocab_index = {tok: i for i, tok in enumerate(vocab)}
    # Use sparse matrix. BOW is ~99% zeros so dense storage is wasteful.
    matrix = lil_matrix((len(token_lists), len(vocab)), dtype=np.int32)
    for i, tokens in enumerate(token_lists):
        counts = Counter(tokens)
        for tok, count in counts.items():
            if tok in vocab_index:
                matrix[i, vocab_index[tok]] = count
    # Convert to CSR format for efficient arithmetic and slicing.
    return csr_matrix(matrix), vocab

In [30]:
def get_top_features_by_ratio(nb_model, vocab, n=20):
    # Log probability of each token given each class.
    log_prob_neg = nb_model.feature_log_prob_[0]
    log_prob_pos = nb_model.feature_log_prob_[1]

    # Likelihood ratio.
    # How much more likely is this token in each class.
    neg_ratio = log_prob_neg - log_prob_pos
    pos_ratio = log_prob_pos - log_prob_neg
    
    # Top tokens most distinctive to each class.
    top_neg_idx = neg_ratio.argsort()[-n:][::-1]
    top_pos_idx = pos_ratio.argsort()[-n:][::-1]
    return {
        0: [vocab[i] for i in top_neg_idx],
        1: [vocab[i] for i in top_pos_idx],
    }

In [10]:
def prepare_model_data(t_col, r_col):
    # Get Agreement-scale row indices for this pairing.
    agreement_idx = get_agreement_index(r_col)
    subset = Survey_df.loc[agreement_idx].copy()
    # Combine lemmas and bigrams as features.
    subset["features"] = subset.apply(
        lambda row: list(row[t_col + "_lemmas"]) + list(row[t_col + "_bigrams"]), axis=1
    )
    # Drop rows with no text, no rating, or neutral rating.
    # We throw out threes. Why? They mean "neither agree nor disagree."
    subset = subset.dropna(subset=[r_col])
    subset = subset[subset["features"].apply(len) > 0]
    subset = subset[subset[r_col] != 3]
    # Binarize rating: above 3 is 1, below 3 is 0 (negative sentiment).
    y = (subset[r_col] > 3).astype(int)
    # Check we have enough data and both classes present.
    if len(y) < 50 or y.nunique() < 2:
        return None, None, None
    return subset, y, subset["features"].tolist()

In [11]:
def split_and_train(features, y):
    # Build BOW from combined lemmas and bigrams.
    X, vocab = build_bow(features)

    # 80/20 stratified split.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=2026, stratify=y
    )

    # Compute vocabulary coverage — tokens seen in training that appear in test.
    train_vocab = set(
        tok for tokens in features[:len(y_train)] for tok in tokens
    )
    test_tokens = set(
        tok for tokens in features[len(y_train):] for tok in tokens
    )
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0
    
    # Train model.
    nb = MultinomialNB()
    nb.fit(X_train, y_train)
    return nb, vocab, X_test, y_train, y_test, vocab_coverage

In [12]:
def split_and_train_undersampled(features, y):
    # Build BOW from combined lemmas and bigrams.
    X, vocab = build_bow(features)

    # Resample only the training set, never the test set.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=2026, stratify=y
    )

    # Compute vocabulary coverage before resampling.
    train_vocab = set(
        tok for tokens in features[:len(y_train)] for tok in tokens
    )
    test_tokens = set(
        tok for tokens in features[len(y_train):] for tok in tokens
    )
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0
    
    # Undersample the majority class in the training set only.
    # Test set is kept as-is to reflect real class distribution.
    rus = RandomUnderSampler(random_state=2026)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)
    # Train model on resampled data.
    nb = MultinomialNB()
    nb.fit(X_train_resampled, y_train_resampled)
    return nb, vocab, X_test, y_train_resampled, y_test, vocab_coverage

In [13]:
# Oversampling experiment.
def split_and_train_oversampled(features, y):
    X, vocab = build_bow(features)
    # Resample only training, never test.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=2026, stratify=y
    )

    train_vocab = set(tok for tokens in features[:len(y_train)] for tok in tokens)
    test_tokens = set(tok for tokens in features[len(y_train):] for tok in tokens)
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0

    # Duplicate minority examples until classes are balanced.
    # Caveat: the model will see the same negative responses many times. Overfitting is possible.
    ros = RandomOverSampler(random_state=2026)
    X_train_resampled, y_train_resampled = ros.fit_resample(X_train, y_train)
    nb = MultinomialNB()
    nb.fit(X_train_resampled, y_train_resampled)
    return nb, vocab, X_test, y_train_resampled, y_test, vocab_coverage

In [14]:
# Similar deal as to the above.
# SMOTE Experiment.
# Not optimistic, of course.
def split_and_train_smote(features, y):
    X, vocab = build_bow(features)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=2026, stratify=y
    )

    train_vocab = set(tok for tokens in features[:len(y_train)] for tok in tokens)
    test_tokens = set(tok for tokens in features[len(y_train):] for tok in tokens)
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0

    # Generate synthetic minority examples by interpolating between existing ones in BOW space.
    # Caveat: interpolating between word count vectors doesn't always produce
    # linguistically meaningful examples. Results may vary.
    smote = SMOTE(random_state=2026)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    nb = MultinomialNB()
    nb.fit(X_train_resampled, y_train_resampled)
    return nb, vocab, X_test, y_train_resampled, y_test, vocab_coverage

In [15]:
# ADASYN Experiment.
def split_and_train_adasyn(features, y):
    X, vocab = build_bow(features)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=2026, stratify=y
    )

    train_vocab = set(tok for tokens in features[:len(y_train)] for tok in tokens)
    test_tokens = set(tok for tokens in features[len(y_train):] for tok in tokens)
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0


    # Adaptive synthetic sampling.
    # More targeted than SMOTE. Still interpolating BOW vectors though, so same caveat applies.
    # Also occasionally throws a fit if the minority class is too small. Thus error handling is included to be super spoofy.
    try:
        adasyn = ADASYN(random_state=2026)
        X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)
    except ValueError as e:
        print(f"ADASYN failed ({e}). SMOTE instead.")
        smote = SMOTE(random_state=42)
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    nb = MultinomialNB()
    nb.fit(X_train_resampled, y_train_resampled)
    return nb, vocab, X_test, y_train_resampled, y_test, vocab_coverage

In [16]:
# SMOTETomek Experiment.

def split_and_train_smotetomek(features, y):
    X, vocab = build_bow(features)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=2026, stratify=y
    )
    train_vocab = set(tok for tokens in features[:len(y_train)] for tok in tokens)
    test_tokens = set(tok for tokens in features[len(y_train):] for tok in tokens)
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0

    # SMOTETomek: oversample the minority AND clean ambiguous majority examples near the boundary.
    # The most sophisticated option here. Addresses both sides of the imbalance problem simultaneously.
    # Caveat: Slow. Sloooow.
    smotetomek = SMOTETomek(random_state=2026)
    X_train_resampled, y_train_resampled = smotetomek.fit_resample(X_train, y_train)
    nb = MultinomialNB()
    nb.fit(X_train_resampled, y_train_resampled)
    return nb, vocab, X_test, y_train_resampled, y_test, vocab_coverage

In [17]:
# Non-class balance handling experiemnt with weights.
def split_and_train_weighted(features, y):
    X, vocab = build_bow(features)
    # Resample only training, never test.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    train_vocab = set(tok for tokens in features[:len(y_train)] for tok in tokens)
    test_tokens = set(tok for tokens in features[len(y_train):] for tok in tokens)
    vocab_coverage = round(
        len(train_vocab & test_tokens) / len(test_tokens) * 100, 1
    ) if test_tokens else 0.0


    # Compute inverse frequency priors.
    # Tell the model that minority class mistakes cost more. 
    neg_count = (y_train == 0).sum()
    pos_count = (y_train == 1).sum()
    total = len(y_train)
    
    # Invert the class frequencies so the rare class gets a higher prior.
    class_prior = [neg_count / total, pos_count / total]
    nb = MultinomialNB(class_prior=class_prior)
    nb.fit(X_train, y_train)
    return nb, vocab, X_test, y_train, y_test, vocab_coverage

In [29]:
# Function to extract performance metrics.
def evaluate_model(nb, vocab, X_test, y_train, y_test, vocab_coverage, t_col, r_col, label, y):
    # Generate predictions.
    y_pred = nb.predict(X_test)
    y_prob = nb.predict_proba(X_test)[:, 1]
    # Compute confusion matrix values.
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    # Record class balance.
    # Take mean of binaries = the portions of ones.
    pos_pct = round(y.mean() * 100, 1)
    neg_pct = round(100 - pos_pct, 1)

    # Return metrics of request.
    return {
        "Label":          label,
        "T_Col":          t_col,
        "R_Col":          r_col,
        "N_Train":        len(y_train),
        "N_Test":         len(y_test),
        "Pos_Pct":        pos_pct,
        "Neg_Pct":        neg_pct,
        "Vocab_Size":     len(vocab),
        "Vocab_Coverage": vocab_coverage,
        "Accuracy":       round(accuracy_score(y_test, y_pred), 3),
        "F1":             round(f1_score(y_test, y_pred), 3),
        "Precision":      round(precision_score(y_test, y_pred), 3),
        "Recall":         round(recall_score(y_test, y_pred), 3),
        "ROC_AUC":        round(roc_auc_score(y_test, y_prob), 3),
        "MCC":            round(matthews_corrcoef(y_test, y_pred), 3),
        "TP":             int(tp),
        "TN":             int(tn),
        "FP":             int(fp),
        "FN":             int(fn),
        "Top_Features":   get_top_features_by_ratio(nb, vocab),
    }

In [19]:
def run_naive_bayes(t_col, r_col, label):
    # Prepare data.
    # Calls above model preparation function to process data.
    subset, y, features = prepare_model_data(t_col, r_col)
    if subset is None:
        print(f"Skipping {label}.")
        return None
    
    # Split, build BOW, and train.
    nb, vocab, X_test, y_train, y_test, vocab_coverage = split_and_train(features, y)

    # Evaluate and return all metrics.
    return evaluate_model(nb, vocab, X_test, y_train, y_test, vocab_coverage, t_col, r_col, label, y)

In [20]:
# Store all primary pairings, with each T_ column predicting its matched R_ column.
Primary_Results = []

for t_col, r_col in Primary_Pairs:
    # Build a readable label for printing and the results table.
    # I.e. delete the stuff that has underscores.
    label = f"{t_col.replace('T_', '')} to {r_col.replace('_encoded', '').replace('R_', '')}."
    print(f"Running: {label}")
    result = run_naive_bayes(t_col, r_col, label)
    if result:
        Primary_Results.append(result)
        # Quick inline summary so we can see progress without waiting for the full table.
        print(f"Accuracy: {result['Accuracy']}. ROC_AUC: {result['ROC_AUC']}")
    print()

Running: Collaboration_Understanding to Collaborative_Activities.
Accuracy: 0.961. ROC_AUC: 0.739

Running: Leader_Performance_Suggestions to Leader_Helps_Understanding.
Accuracy: 0.938. ROC_AUC: 0.682

Running: Leader_Performance_Suggestions to Leader_Subject_Competence.
Accuracy: 0.973. ROC_AUC: 0.633

Running: Leader_Performance_Suggestions to Leader_Has_Plan.
Accuracy: 0.98. ROC_AUC: 0.607

Running: Leader_Performance_Suggestions to Leader_Willing_To_Help.
Accuracy: 0.975. ROC_AUC: 0.571

Running: Leader_Performance_Suggestions to Leader_Average.
Accuracy: 0.969. ROC_AUC: 0.63

Running: Other_Suggestions to Overall_Session_Helpfulness.
Accuracy: 0.846. ROC_AUC: 0.852

Running: Program_Overall_Suggestions to Overall_Session_Helpfulness.
Accuracy: 0.93. ROC_AUC: 0.642



In [21]:
# Store all primary pairings with undersampling applied to training set.
Primary_Results_Undersampled = []

for t_col, r_col in Primary_Pairs:
    # Build a readable label for printing and the results table.
    label = f"{t_col.replace('T_', '')} to {r_col.replace('_encoded', '').replace('R_', '')}."
    print(f"Running: {label}")
    # Prepare data same as before -- filter, featurize, binarize.
    subset, y, features = prepare_model_data(t_col, r_col)
    if subset is None:
        print(f"Skipping {label}.")
        continue
    # Use undersampled split and train instead of standard split.
    nb, vocab, X_test, y_train, y_test, vocab_coverage = split_and_train_undersampled(features, y)
    result = evaluate_model(nb, vocab, X_test, y_train, y_test, vocab_coverage, t_col, r_col, label, y)
    if result:
        Primary_Results_Undersampled.append(result)
        # Quick inline summary so we can see progress without waiting for the full table.
        print(f"Accuracy: {result['Accuracy']}. ROC_AUC: {result['ROC_AUC']}. MCC: {result['MCC']}")
    print()

Running: Collaboration_Understanding to Collaborative_Activities.
Accuracy: 0.855. ROC_AUC: 0.776. MCC: 0.231

Running: Leader_Performance_Suggestions to Leader_Helps_Understanding.
Accuracy: 0.607. ROC_AUC: 0.839. MCC: 0.235

Running: Leader_Performance_Suggestions to Leader_Subject_Competence.
Accuracy: 0.571. ROC_AUC: 0.836. MCC: 0.144

Running: Leader_Performance_Suggestions to Leader_Has_Plan.
Accuracy: 0.599. ROC_AUC: 0.784. MCC: 0.117

Running: Leader_Performance_Suggestions to Leader_Willing_To_Help.
Accuracy: 0.597. ROC_AUC: 0.785. MCC: 0.133

Running: Leader_Performance_Suggestions to Leader_Average.
Accuracy: 0.547. ROC_AUC: 0.846. MCC: 0.153

Running: Other_Suggestions to Overall_Session_Helpfulness.
Accuracy: 0.664. ROC_AUC: 0.838. MCC: 0.302

Running: Program_Overall_Suggestions to Overall_Session_Helpfulness.
Accuracy: 0.619. ROC_AUC: 0.78. MCC: 0.205



In [31]:
# Map technique names to their split/train functions.
# Meta-trainer that stores all results in Results_By_Technique.
Techniques = {
    "Original":     split_and_train,
    "Undersampled": split_and_train_undersampled,
    "Oversampled":  split_and_train_oversampled,
    "SMOTE":        split_and_train_smote,
    "ADASYN":       split_and_train_adasyn,
    'SMOTETomek':   split_and_train_smotetomek,
    "Weighted":     split_and_train_weighted,
}

# Store results per technique.
Results_By_Technique = {}

for technique_name, split_fn in Techniques.items():
    print(f"Technique: {technique_name}")
    technique_results = []
    for t_col, r_col in Primary_Pairs:
        label = f"{t_col.replace('T_', '')} to {r_col.replace('_encoded', '').replace('R_', '')}."
        print(f"  {label}")
        subset, y, features = prepare_model_data(t_col, r_col)
        if subset is None:
            print(f"  Skipping {label}.")
            continue
        nb, vocab, X_test, y_train, y_test, vocab_coverage = split_fn(features, y)
        result = evaluate_model(nb, vocab, X_test, y_train, y_test, vocab_coverage, t_col, r_col, label, y)
        if result:
            technique_results.append(result)
            print(f"  Accuracy: {result['Accuracy']}. ROC_AUC: {result['ROC_AUC']}. MCC: {result['MCC']}")
    Results_By_Technique[technique_name] = technique_results
    print()

Technique: Original
  Collaboration_Understanding to Collaborative_Activities.
  Accuracy: 0.961. ROC_AUC: 0.739. MCC: 0.091
  Leader_Performance_Suggestions to Leader_Helps_Understanding.
  Accuracy: 0.938. ROC_AUC: 0.682. MCC: 0.193
  Leader_Performance_Suggestions to Leader_Subject_Competence.
  Accuracy: 0.973. ROC_AUC: 0.633. MCC: 0.059
  Leader_Performance_Suggestions to Leader_Has_Plan.
  Accuracy: 0.98. ROC_AUC: 0.607. MCC: 0.032
  Leader_Performance_Suggestions to Leader_Willing_To_Help.
  Accuracy: 0.975. ROC_AUC: 0.571. MCC: 0.02
  Leader_Performance_Suggestions to Leader_Average.
  Accuracy: 0.969. ROC_AUC: 0.63. MCC: 0.118
  Other_Suggestions to Overall_Session_Helpfulness.
  Accuracy: 0.846. ROC_AUC: 0.852. MCC: 0.256
  Program_Overall_Suggestions to Overall_Session_Helpfulness.
  Accuracy: 0.93. ROC_AUC: 0.642. MCC: 0.116

Technique: Undersampled
  Collaboration_Understanding to Collaborative_Activities.
  Accuracy: 0.855. ROC_AUC: 0.776. MCC: 0.231
  Leader_Performance_

In [32]:
# Build unified comparison table across all techniques and pairings.
# Top_Features excluded.
Comparison_Rows = []

for technique_name, results in Results_By_Technique.items():
    for r in results:
        Comparison_Rows.append({
            "Technique":      technique_name,
            "Pairing":        r["Label"],
            "N":              r["N_Train"] + r["N_Test"],
            "Vocab Size":     r["Vocab_Size"],
            "Pos %":          r["Pos_Pct"],
            "Neg %":          r["Neg_Pct"],
            "Accuracy":       r["Accuracy"],
            "F1":             r["F1"],
            "Precision":      r["Precision"],
            "Recall":         r["Recall"],
            "ROC AUC":        r["ROC_AUC"],
            "MCC":            r["MCC"],
            "Vocab Coverage": r["Vocab_Coverage"],
        })

Comparison_df = pd.DataFrame(Comparison_Rows)

In [33]:
# Print full comparison table sorted by ROC AUC descending.
print(Comparison_df.sort_values("ROC AUC", ascending=False).to_string(index=False))

   Technique                                                       Pairing     N  Vocab Size  Pos %  Neg %  Accuracy    F1  Precision  Recall  ROC AUC   MCC  Vocab Coverage
      ADASYN             Other_Suggestions to Overall_Session_Helpfulness.  1210        3408   88.3   11.7     0.738 0.831      0.970   0.727    0.867 0.371            36.7
       SMOTE             Other_Suggestions to Overall_Session_Helpfulness.  1193        3408   88.3   11.7     0.745 0.836      0.970   0.735    0.865 0.378            36.7
  SMOTETomek             Other_Suggestions to Overall_Session_Helpfulness.  1193        3408   88.3   11.7     0.745 0.836      0.970   0.735    0.865 0.378            36.7
    Original             Other_Suggestions to Overall_Session_Helpfulness.   741        3408   88.3   11.7     0.846 0.913      0.916   0.909    0.852 0.256            36.7
Undersampled             Leader_Performance_Suggestions to Leader_Average.  5248       54221   97.3    2.7     0.547 0.698      0.996  